# Variations on gradient descent

**Outcomes**

- Understand the issues with gradient descent
- Understand the concepts of momentum and adaptive learning rates
- Understand the concept of epochs and batches
- Be familiar with GD variants like Adagrad, RMSprop, and ADAM


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

import os
from urllib.request import urlretrieve


In [ ]:
def plot_surf(f, **kw):
    L, n = 2, 400
    x = np.linspace(-L, L, n)
    y = x.copy()
    X, Y = np.meshgrid(x, y)

    # set up 3d plot
    fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': '3d'})

    Z = f([X, Y])

    ax.plot_surface(X, Y, Z, cmap="viridis", **kw)
    return ax


def get_trace_xyz(f, trace):
    xy = [i["x"] for i in trace]
    x, y = zip(*xy)
    z = f([np.array(x), np.array(y)])
    return x, y, z


def plot_path(f, trace, **kw):
    ax = plot_surf(f, **kw)
    x, y, z = get_trace_xyz(f, trace)
    ax.scatter3D(x, y, z, c="red")
    ax.plot(x, y, z, c="red")
    ax.view_init(10, 300)
    return ax


def plot_contour_path(f, trace, ax=None):
    L, n = 2, 400
    x = np.linspace(-L, L, n)
    y = x.copy()
    X, Y = np.meshgrid(x, y)
    Z = f([X, Y])

    # set up plot
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    CS = ax.contour(X, Y, Z)
    ax.clabel(CS, inline=True, fontsize=10)
    
    x, y, z = get_trace_xyz(f, trace)
    ax.scatter(x, y, c=np.linspace(0.5, 1, len(x)), s=8)
    ax.set_title("Convergence in {} iterations".format(len(x)));
    return ax

In [ ]:
def grad_desc_torch(f, x0, epsilon=1e-3, T=200, alpha=0.1):
    trace = []
    # x is a "leaf" tensor, which is what we want
    x = torch.tensor(x0, requires_grad=True)

    for i in range(T):
        # 1. Zero the gradient from the previous step
        #    Do this BEFORE the forward pass
        if x.grad is not None:
            x.grad.zero_()

        # 2. Forward pass: compute the function value
        fx = f(x)

        # 3. Backward pass: compute the gradient
        fx.backward()
        dfdx = x.grad
        err = max(abs(dfdx))
        
        # --- Store results before updating x ---
        status = dict(
            i=i,
            err=err.item(),
            x=x.clone().detach().numpy(),
            fx=fx.item(),
            dfdx=dfdx.clone().detach().numpy(),
        )
        trace.append(status)

        if err < epsilon:
            return trace

        # 4. Update step: modify x without tracking it in the graph
        with torch.no_grad():
            x -= alpha * dfdx

    raise ValueError("No convergence")

## Local Minimum Problem

- Gradient descent can be used to find a local minimum near the starting position
- However, sometimes our starting position is close to a local minimum, but far from a global or better "less-local" minimum
- For example, consider this objective function surface:

In [ ]:
def check_tensor(x):
    if isinstance(x, torch.Tensor):
        return x
    if isinstance(x, list):
        return check_tensor(np.array(x))
    return torch.tensor(x)

def f2(x):
    xt = check_tensor(x)
    return  -torch.sinc(xt[0])-2*torch.exp(-((xt[0]-1)**2 + (xt[1]-1)**2))

ax = plot_surf(f2, alpha=0.4)
ax.view_init(10, 30);

### Gradient Descent Stuck

- We will try to If we try to apply gradient descent to this surface 
- We'll choose a starting point of `[1.2, -1.9]`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-4, 4, 100)
fig, ax = plt.subplots()
ax.plot(x, x**2)
ax.plot(x, np.abs(x))

In [ ]:
trace1 = grad_desc_torch(f2, [1.2, -1.9], T=5000);
ax = plot_path(f2, trace1, alpha=0.4)
ax.view_init(10, 30);
plot_contour_path(f2, trace1);

### Need for Momentum

- From contour plot above we see we move very little in `x2` dimension

In [ ]:
trace1[0]["x"] - trace1[-1]["x"]

- However, we do move a little
- Idea: could we continue to propagate movement in `x2` dimension to avoid getting stuck?
- Alternative idea: could we "magnify" movements in x2 dimension where gradients are not as steep? (keep in mind -- will revisit)

### Momentum

- Let's explore continuing to propogate movement in 
- Intuition: build up momentum in each direction
    - Will allow us to pass over "flat" spaces in dimensions with very small gradients
    - Will also dampen movements in steep dimensions
- Application: use exponentially decaying moving average to track momentum in each dimension
    - Keep track of momentum on iteration $i$ using $v_i$
    - Start with $v_0 = 0$
    - Update rule for iteration $i$: $$\begin{align*}v_{i+1} &= \gamma v_i + \alpha \nabla f(x) \\ x_{i+1} &= x_i - v_{i+1}\end{align*}$$
    

#### Code example

In [ ]:
def grad_desc_torch_mom(f, x0, epsilon=1e-3, T=200, alpha=0.1, gamma=0.7):

    trace = []
    x = torch.tensor(x0, requires_grad=True)
    v = torch.zeros(x.shape)  # NEW CODE
    for i in range(T):
        fx = f(x)
        fx.backward()
        dfdx = x.grad
        vp = alpha*dfdx + gamma*v  # NEW CODE
        xp = x - vp  # NEW CODE
        err = max(abs(dfdx))
        status = dict(
            i=i,
            fx=fx.item(),
            dfdx=dfdx.numpy(),
            err=err.item(),
            x=x.clone().detach().numpy(),
        )
        trace.append(status)
        if err < epsilon and max(abs(vp)) < epsilon:
            return trace

        x = xp.clone().detach().requires_grad_(True)
        v = vp.clone()  # NEW CODE

    print("FAILED TO CONVERGE")
    return trace

#### Application

- Let's try this momentum algorithm with our troublesome surface from above
- Note we'll use the same learning rate and starting point

In [ ]:
trace_mom = grad_desc_torch_mom(f2, [1.2, -1.9], T=1000);
ax = plot_path(f2, trace_mom, alpha=0.3)
ax.view_init(10, 30);
plot_contour_path(f2, trace_mom);

- We no longer get stuck!
- The small momentum built up in dimension `x2` allowed the algorithm to continue moving down that very slight slope until we find the global minimum

In [ ]:
trace_mom[0]["x"] - trace_mom[-1]["x"]

## Paper review

- Gradient descent algorithms come in many variants
- We've seen one example here that adds momentum
- There are many others
- There is an excellent review paper that describes the algorithms
- **Reference**
    - paper: https://arxiv.org/abs/1609.04747
    - website: https://ruder.io/optimizing-gradient-descent/